In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

# 1. Import from the local diffusion module
from diffusion import (
    Diffusion, 
    DiffusionConfig, 
    train_diffusion_model, 
    generate_images,
    sinusoidal_embedding # Importing your sinusoidal helper for better time scaling
)

# ==========================================
# SMART PARAMETER CONFIGURATION
# ==========================================
# Data Params
IMAGE_SIZE = 28
CHANNELS = 1
BATCH_SIZE = 128      # 128 balances gradient stability and memory for a small dataset.
TARGET_DIGIT = 6      # The specific MNIST digit we want to model.

# Model Params
BASE_DIM = 64         # Base channel dimension for the medium U-Net.
TIME_DIM = 128        # Explicit time embedding dimension for better step conditioning.

# Training Params
EPOCHS = 7        # ~7-10 epochs handles this digit subset nicely.
LEARNING_RATE = 1e-3  # 1e-3 is ideal for Adam on this scale.

# Diffusion Process Params
BETA_MIN = 0.01        # Standard continuous-time lower bound for noise.
BETA_MAX = 20.0       # Standard continuous-time upper bound to reach pure Gaussian.
NUM_STEPS = 1000      # 1000 steps ensure smooth transitions during the reverse ODE/SDE.
NUM_GENERATIONS = 16  # For the 4x4 grid plot.
# ==========================================

# 2. Define the Custom Medium-Sized U-Net Network
class ConvBlock(nn.Module):
    """A basic Convolutional block with Time conditioning and a residual connection."""
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act = nn.GELU()

    def forward(self, x, t_emb):
        h = self.act(self.conv1(x))
        h = h + self.time_proj(t_emb)[..., None, None]
        h = self.act(self.conv2(h))
        return h + self.skip(x)

class UNetMedium(nn.Module):
    """
    A medium-sized U-Net with skip connections, downsampling, bottleneck, and upsampling.
    """
    def __init__(self, in_channels=CHANNELS, base_dim=BASE_DIM, time_dim=TIME_DIM):
        super().__init__()
        self.time_dim = time_dim
        
        # Time embedding MLP
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.GELU(),
            nn.Linear(time_dim * 4, time_dim)
        )
        
        self.inc = nn.Conv2d(in_channels, base_dim, 3, padding=1)
        
        # Downsampling path (spatial dim halved with maxpool, channels doubled in block)
        self.down1 = ConvBlock(base_dim, base_dim, time_dim)
        self.down2 = ConvBlock(base_dim, base_dim * 2, time_dim)
        self.down3 = ConvBlock(base_dim * 2, base_dim * 4, time_dim)
        
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck (Mid)
        self.mid = ConvBlock(base_dim * 4, base_dim * 4, time_dim)
        
        # Upsampling path
        self.up1 = nn.ConvTranspose2d(base_dim * 4, base_dim * 2, 2, stride=2)
        self.up_block1 = ConvBlock(base_dim * 4, base_dim * 2, time_dim) # in_ch = base*2(up) + base*2(skip) = base*4
        
        self.up2 = nn.ConvTranspose2d(base_dim * 2, base_dim, 2, stride=2)
        self.up_block2 = ConvBlock(base_dim * 2, base_dim, time_dim) # in_ch = base(up) + base(skip) = base*2
        
        self.outc = nn.Conv2d(base_dim, in_channels, 3, padding=1)

    def forward(self, x, t):
        # Time embeddings
        t_emb = sinusoidal_embedding(t, self.time_dim) 
        t_emb = self.time_mlp(t_emb)
        
        # Initial 
        x0 = self.inc(x) # 28x28
        
        # Down 1
        d1 = self.down1(x0, t_emb) # 28x28
        p1 = self.pool(d1)         # 14x14
        
        # Down 2
        d2 = self.down2(p1, t_emb) # 14x14
        p2 = self.pool(d2)         # 7x7
        
        # Down 3
        d3 = self.down3(p2, t_emb) # 7x7
        
        # Mid
        m = self.mid(d3, t_emb)    # 7x7
        
        # Up 1
        u1 = self.up1(m)           # 14x14
        u1 = torch.cat([u1, d2], dim=1) # Concat skip connection
        u1 = self.up_block1(u1, t_emb)  # 14x14
        
        # Up 2
        u2 = self.up2(u1)          # 28x28
        u2 = torch.cat([u2, d1], dim=1) # Concat skip connection
        u2 = self.up_block2(u2, t_emb)  # 28x28
        
        # Output
        return self.outc(u2)

# 3. Setup Dataset and DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # Crucial: Maps [0, 1] to [-1, 1]
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Filter dataset to only include the TARGET_DIGIT
idx_target = mnist_train.targets == TARGET_DIGIT
custom_dataset = Subset(mnist_train, torch.where(idx_target)[0])
loader = DataLoader(custom_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 4. Apply Configuration via your Dataclass
cfg = DiffusionConfig(
    image_size=IMAGE_SIZE,
    channels=CHANNELS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS, 
    lr=LEARNING_RATE, 
    beta_min=BETA_MIN,
    beta_max=BETA_MAX,
    num_steps=NUM_STEPS,
    num_sample_grid=NUM_GENERATIONS,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 5. Initialize Model and Diffusion Utilities
diffusion_process = Diffusion(
    beta_min=cfg.beta_min, 
    beta_max=cfg.beta_max, 
    num_steps=cfg.num_steps, 
    device=cfg.device
)

# Use the new Medium U-Net
model = UNetMedium(
    in_channels=cfg.channels, 
    base_dim=BASE_DIM, 
    time_dim=TIME_DIM
).to(cfg.device)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

# 6. Train the Model
print(f"Starting training on device: {cfg.device}...")
print(f"Dataset size: {len(custom_dataset)} images of digit '{TARGET_DIGIT}'")

results = train_diffusion_model(
    cfg=cfg,
    diffusion=diffusion_process,
    model=model,
    loader=loader,
    optimizer=optimizer
)
print("Training complete!")

# 7. Generate and display images
print(f"Generating {cfg.num_sample_grid} sample images...")
samples = generate_images(cfg, diffusion_process, model, num_images=cfg.num_sample_grid)

# Plotting the generated images
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flatten()):
    # Re-normalize from [-1, 1] back to [0, 1] for displaying
    img = (samples[i].cpu().squeeze() + 1) / 2
    ax.imshow(img.clamp(0, 1), cmap='gray')
    ax.axis('off')
    
plt.suptitle(f"Generated MNIST Digit '{TARGET_DIGIT}'")
plt.tight_layout()
plt.show()

In [ ]:
# 7. Generate and display images
print(f"Generating {cfg.num_sample_grid} sample images...")
samples = generate_images(cfg, diffusion_process, model, num_images=cfg.num_sample_grid)

# Plotting the generated images
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flatten()):
    # Re-normalize from [-1, 1] back to [0, 1] for displaying
    img = (samples[i].cpu().squeeze() + 1) / 2
    ax.imshow(img.clamp(0, 1), cmap='gray')
    ax.axis('off')
    
plt.suptitle(f"Generated MNIST Digit '{TARGET_DIGIT}'")
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

# 1. Import from the local diffusion module
from diffusion import (
    Diffusion, 
    DiffusionConfig, 
    train_diffusion_model, 
    generate_images,
    sinusoidal_embedding # Importing your sinusoidal helper for better time scaling
)

# ==========================================
# SMART PARAMETER CONFIGURATION
# ==========================================
# Data Params
IMAGE_SIZE = 28
CHANNELS = 1
BATCH_SIZE = 128      # 128 balances gradient stability and memory for a small dataset.
TARGET_DIGIT = 6      # The specific MNIST digit we want to model.

# Model Params
BASE_DIM = 64         # Base channel dimension for the medium U-Net.
TIME_DIM = 128        # Explicit time embedding dimension for better step conditioning.

# Training Params
EPOCHS = 7        # ~7-10 epochs handles this digit subset nicely.
LEARNING_RATE = 1e-3  # 1e-3 is ideal for Adam on this scale.

# Diffusion Process Params
BETA_MIN = 0.01        # Standard continuous-time lower bound for noise.
BETA_MAX = 20.0       # Standard continuous-time upper bound to reach pure Gaussian.
NUM_STEPS = 1000      # 1000 steps ensure smooth transitions during the reverse ODE/SDE.
NUM_GENERATIONS = 16  # For the 4x4 grid plot.
# ==========================================

# 2. Define the Custom Medium-Sized U-Net Network
class ConvBlock(nn.Module):
    """A basic Convolutional block with Time conditioning and a residual connection."""
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act = nn.GELU()

    def forward(self, x, t_emb):
        h = self.act(self.conv1(x))
        h = h + self.time_proj(t_emb)[..., None, None]
        h = self.act(self.conv2(h))
        return h + self.skip(x)

class UNetMedium(nn.Module):
    """
    A medium-sized U-Net with skip connections, downsampling, bottleneck, and upsampling.
    """
    def __init__(self, in_channels=CHANNELS, base_dim=BASE_DIM, time_dim=TIME_DIM):
        super().__init__()
        self.time_dim = time_dim
        
        # Time embedding MLP
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.GELU(),
            nn.Linear(time_dim * 4, time_dim)
        )
        
        self.inc = nn.Conv2d(in_channels, base_dim, 3, padding=1)
        
        # Downsampling path (spatial dim halved with maxpool, channels doubled in block)
        self.down1 = ConvBlock(base_dim, base_dim, time_dim)
        self.down2 = ConvBlock(base_dim, base_dim * 2, time_dim)
        self.down3 = ConvBlock(base_dim * 2, base_dim * 4, time_dim)
        
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck (Mid)
        self.mid = ConvBlock(base_dim * 4, base_dim * 4, time_dim)
        
        # Upsampling path
        self.up1 = nn.ConvTranspose2d(base_dim * 4, base_dim * 2, 2, stride=2)
        self.up_block1 = ConvBlock(base_dim * 4, base_dim * 2, time_dim) # in_ch = base*2(up) + base*2(skip) = base*4
        
        self.up2 = nn.ConvTranspose2d(base_dim * 2, base_dim, 2, stride=2)
        self.up_block2 = ConvBlock(base_dim * 2, base_dim, time_dim) # in_ch = base(up) + base(skip) = base*2
        
        self.outc = nn.Conv2d(base_dim, in_channels, 3, padding=1)

    def forward(self, x, t):
        # Time embeddings
        t_emb = sinusoidal_embedding(t, self.time_dim) 
        t_emb = self.time_mlp(t_emb)
        
        # Initial 
        x0 = self.inc(x) # 28x28
        
        # Down 1
        d1 = self.down1(x0, t_emb) # 28x28
        p1 = self.pool(d1)         # 14x14
        
        # Down 2
        d2 = self.down2(p1, t_emb) # 14x14
        p2 = self.pool(d2)         # 7x7
        
        # Down 3
        d3 = self.down3(p2, t_emb) # 7x7
        
        # Mid
        m = self.mid(d3, t_emb)    # 7x7
        
        # Up 1
        u1 = self.up1(m)           # 14x14
        u1 = torch.cat([u1, d2], dim=1) # Concat skip connection
        u1 = self.up_block1(u1, t_emb)  # 14x14
        
        # Up 2
        u2 = self.up2(u1)          # 28x28
        u2 = torch.cat([u2, d1], dim=1) # Concat skip connection
        u2 = self.up_block2(u2, t_emb)  # 28x28
        
        # Output
        return self.outc(u2)

# 3. Setup Dataset and DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # Crucial: Maps [0, 1] to [-1, 1]
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Filter dataset to only include the TARGET_DIGIT
idx_target = mnist_train.targets == TARGET_DIGIT
custom_dataset = Subset(mnist_train, torch.where(idx_target)[0])
loader = DataLoader(custom_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 4. Apply Configuration via your Dataclass
cfg = DiffusionConfig(
    image_size=IMAGE_SIZE,
    channels=CHANNELS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS, 
    lr=LEARNING_RATE, 
    beta_min=BETA_MIN,
    beta_max=BETA_MAX,
    num_steps=NUM_STEPS,
    num_sample_grid=NUM_GENERATIONS,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 5. Initialize Model and Diffusion Utilities
diffusion_process = Diffusion(
    beta_min=cfg.beta_min, 
    beta_max=cfg.beta_max, 
    num_steps=cfg.num_steps, 
    device=cfg.device
)

# Use the new Medium U-Net
model = UNetMedium(
    in_channels=cfg.channels, 
    base_dim=BASE_DIM, 
    time_dim=TIME_DIM
).to(cfg.device)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

# 6. Train the Model
print(f"Starting training on device: {cfg.device}...")
print(f"Dataset size: {len(custom_dataset)} images of digit '{TARGET_DIGIT}'")

results = train_diffusion_model(
    cfg=cfg,
    diffusion=diffusion_process,
    model=model,
    loader=loader,
    optimizer=optimizer
)
print("Training complete!")

# 7. Generate and display images
print(f"Generating {cfg.num_sample_grid} sample images...")
samples = generate_images(cfg, diffusion_process, model, num_images=cfg.num_sample_grid)

# Plotting the generated images
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flatten()):
    # Re-normalize from [-1, 1] back to [0, 1] for displaying
    img = (samples[i].cpu().squeeze() + 1) / 2
    ax.imshow(img.clamp(0, 1), cmap='gray')
    ax.axis('off')
    
plt.suptitle(f"Generated MNIST Digit '{TARGET_DIGIT}'")
plt.tight_layout()
plt.show()